In [14]:
import sys
import os
import psycopg2
from dotenv import load_dotenv
import QuantLib as ql


sys.path.append(os.path.abspath(".."))
from utils.black_scholes import get_implied_vol

load_dotenv()

url = os.getenv("DATABASE_URL")

In [15]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [16]:
postfix_list = [
    "CD6D", #04-22
]
expiry_date = ql.Date(22, 4, 2026)
eval_date = ql.Date(16, 4, 2026)
option_typy = "call"

ticker = "SR"
min_strike = 270
max_strike = 370
strike_step = 10

strike = min_strike
tickers = []
while strike < max_strike:
    for postfix in postfix_list:
        option_ticker = ticker + str(strike) + postfix
        tickers.append(option_ticker)
    strike += strike_step
print(tickers)

['SR270CD6D', 'SR280CD6D', 'SR290CD6D', 'SR300CD6D', 'SR310CD6D', 'SR320CD6D', 'SR330CD6D', 'SR340CD6D', 'SR350CD6D', 'SR360CD6D']


In [17]:
query = """
SELECT DISTINCT ON (ticker)
    ticker,
    bids,
    asks
FROM orderbooks
WHERE ticker = ANY(%s)
ORDER BY ticker, timestamp DESC;
"""

cur.execute(query, (tickers,))
rows = cur.fetchall()

In [18]:
rows

[('SR270CD6D',
  [{'price': 52.82, 'quantity': 800}],
  [{'price': 54.22, 'quantity': 800}]),
 ('SR280CD6D',
  [{'price': 42.7, 'quantity': 800}],
  [{'price': 44.19, 'quantity': 800}]),
 ('SR290CD6D',
  [{'price': 32.79, 'quantity': 800},
   {'price': 32.44, 'quantity': 800},
   {'price': 32.36, 'quantity': 800},
   {'price': 10.15, 'quantity': 10}],
  [{'price': 33.13, 'quantity': 800},
   {'price': 33.96, 'quantity': 800},
   {'price': 34.26, 'quantity': 800}]),
 ('SR300CD6D',
  [{'price': 22.82, 'quantity': 800},
   {'price': 22.6, 'quantity': 800},
   {'price': 22.38, 'quantity': 800},
   {'price': 20.39, 'quantity': 800},
   {'price': 0.02, 'quantity': 110}],
  [{'price': 23.16, 'quantity': 800},
   {'price': 23.66, 'quantity': 200},
   {'price': 23.89, 'quantity': 800},
   {'price': 24.28, 'quantity': 800}]),
 ('SR310CD6D',
  [{'price': 12.73, 'quantity': 2500},
   {'price': 12.72, 'quantity': 800},
   {'price': 12.47, 'quantity': 800},
   {'price': 11.67, 'quantity': 800},
   {

In [19]:
def compute_mid_price(bids, asks):
    best_bid = None
    best_ask = None
    
    if bids:
        best_bid = max(bids, key=lambda x: x["price"])["price"]
    if asks:
        best_ask = min(asks, key=lambda x: x["price"])["price"]

    if best_bid is not None and best_ask is not None:
        return (best_bid + best_ask) / 2
    return best_bid or best_ask

In [20]:
result = []

for ticker, bids, asks in rows:
    mid = compute_mid_price(bids, asks)

    result.append({
        "ticker": ticker,
        "mid": mid,
    })
result

[{'ticker': 'SR270CD6D', 'mid': 53.519999999999996},
 {'ticker': 'SR280CD6D', 'mid': 43.445},
 {'ticker': 'SR290CD6D', 'mid': 32.96},
 {'ticker': 'SR300CD6D', 'mid': 22.990000000000002},
 {'ticker': 'SR310CD6D', 'mid': 13.315000000000001},
 {'ticker': 'SR320CD6D', 'mid': 4.465},
 {'ticker': 'SR330CD6D', 'mid': 0.5700000000000001},
 {'ticker': 'SR340CD6D', 'mid': 0.09},
 {'ticker': 'SR350CD6D', 'mid': 0.06},
 {'ticker': 'SR360CD6D', 'mid': 0.55}]

In [23]:
data = result
spot_price = 322
risk_free_rate = 0.15

result = []

for r in data:
    ticker = r["ticker"]
    mid = r["mid"]
    strike = int(r["ticker"][2:5])


    if strike is None:
        continue

    try:
        iv = get_implied_vol(
            market_price=mid,
            spot_price=spot_price,
            strike_price=strike,
            risk_free_rate=risk_free_rate,
            expiry_date=expiry_date,
            eval_date=eval_date,
            option_type=option_type
            )
    except Exception as e:
        iv = None
        print(e)
        

    result.append({
        "strike": strike,
        "mid": mid,
        "iv": iv,
        "ticker": ticker
    })
result

[{'strike': 270,
  'mid': 53.519999999999996,
  'iv': 0.8919256995360423,
  'ticker': 'SR270CD6D'},
 {'strike': 280,
  'mid': 43.445,
  'iv': 0.7236281658262674,
  'ticker': 'SR280CD6D'},
 {'strike': 290,
  'mid': 32.96,
  'iv': 0.45981572427173434,
  'ticker': 'SR290CD6D'},
 {'strike': 300,
  'mid': 22.990000000000002,
  'iv': 0.3377688495681583,
  'ticker': 'SR300CD6D'},
 {'strike': 310,
  'mid': 13.315000000000001,
  'iv': 0.2570474838633749,
  'ticker': 'SR310CD6D'},
 {'strike': 320,
  'mid': 4.465,
  'iv': 0.17423473813838397,
  'ticker': 'SR320CD6D'},
 {'strike': 330,
  'mid': 0.5700000000000001,
  'iv': 0.16930471604329125,
  'ticker': 'SR330CD6D'},
 {'strike': 340, 'mid': 0.09, 'iv': 0.209517463724393, 'ticker': 'SR340CD6D'},
 {'strike': 350, 'mid': 0.06, 'iv': 0.2866979147784779, 'ticker': 'SR350CD6D'},
 {'strike': 360, 'mid': 0.55, 'iv': 0.5341099445295882, 'ticker': 'SR360CD6D'}]